# The Course Benchmark Table

**DMML - Data Mining & Machine Learning**

This notebook is not a weekly assignment. It is the one place where the results
of every week come together.

Each weekly notebook ends with a single `save_results(...)` call that appends its
model scores to a shared file, `benchmark_results.csv`. This notebook reads that
file and builds the comparison table.

**How to use it**

1. Finish a weekly notebook and run its final *Save to the course benchmark* cell.
2. Come back here and run this notebook top to bottom.
3. Watch the table grow: new weeks add rows, new models add columns.

You will run this notebook about ten times over the semester. The interesting
part is not any single run, it is the difference between runs.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from dmml_benchmark import (
    BENCHMARK_PATH,
    clear_benchmark,
    coverage,
    load_benchmark,
    pivot_wide,
    repeated_measurements,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("Reading:", BENCHMARK_PATH)
print("Exists:", BENCHMARK_PATH.exists())

## 1. The Long Table

Long format is the storage format: one row per (model, metric) measurement. It is
boring to read and easy to append to, which is exactly what you want from a file
that grows for ten weeks.

In [ ]:
benchmark_long = load_benchmark()

if benchmark_long.empty:
    print("No results recorded yet.")
    print("Run the final save cell of any weekly notebook, then re-run this one.")
else:
    print(f"{len(benchmark_long)} rows, {benchmark_long['week'].nunique()} week(s), "
          f"{benchmark_long['model'].nunique()} distinct models")

benchmark_long.tail(15)

## 2. Coverage

Which weeks have reported in, and which are still missing.

In [ ]:
coverage(benchmark_long)

## 3. The Wide View

Wide format is the reading format: each model becomes a column, so one row lets
you scan across every model that has ever been run on that dataset and metric.

This is where the point of the whole exercise shows up. Wine is scored in Week 03
and again in Week 05; Digits in Week 04 and again in Week 08. Those weeks land on
the *same row*, months apart, and you can read straight across to see whether the
newer, heavier method actually won.

One table per dataset, deliberately. A forecasting RMSE and a clustering
silhouette are not competing for the same prize.

**Comparability rule.** Two scores may be compared only if they share the same
task, target, split, and metric. Everything else is a coincidence of formatting.

In [ ]:
if benchmark_long.empty:
    print("Nothing to pivot yet.")
else:
    for (task, dataset), part in benchmark_long.groupby(["task_type", "dataset"]):
        wide = pivot_wide(part).drop(columns=["dataset", "task_type"])
        wide = wide.dropna(axis=1, how="all")
        weeks = ", ".join(sorted(part["week"].unique()))
        print(f"\n=== {dataset} - {task} (from {weeks}) ".ljust(90, "="))
        display(wide)

### Same Model, Two Weeks

A baseline recorded twice on the same dataset and split should give exactly the
same number. When it does not, one of the two runs was not what it claimed to be
- a different seed, a different scaler, a different split. The wide view above
keeps only the first value, so it will not tell you about the disagreement.

In [ ]:
repeats = repeated_measurements(benchmark_long)
if repeats.empty:
    print("No model has been measured twice on the same setup yet.")
else:
    display(repeats)
    if not repeats["agrees"].all():
        print("Some repeats disagree. Check the split and random_state in those weeks.")

## 4. Progress Across Weeks

For each task type and metric, the best score recorded in each week. This is the
picture of the course: sometimes the new method wins convincingly, sometimes it
barely beats a baseline you wrote in week 3.

In [ ]:
if benchmark_long.empty:
    print("Nothing to plot yet.")
else:
    best = (
        benchmark_long
        .groupby(["task_type", "metric", "week"], as_index=False)["score"]
        .max()
    )
    groups = list(best.groupby(["task_type", "metric"]))
    n = len(groups)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.4 * nrows), squeeze=False)

    for ax, ((task, metric), part) in zip(axes.ravel(), groups):
        part = part.sort_values("week")
        ax.bar(part["week"], part["score"], color="steelblue")
        ax.set_title(f"{task}\nbest {metric}", fontsize=10)
        ax.tick_params(axis="x", labelrotation=45)
    for ax in axes.ravel()[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 5. Questions To Ask Every Time You Re-Run This

Answer these in a sentence or two each time you add a week. Keep your answers in
this notebook; they become your revision notes.

1. Did this week's method beat the best previous model on a comparable row? By
   how much?
2. Did it beat the *dummy* baseline by a margin that would matter to someone
   using the model?
3. What did the new method cost - training time, tuning effort, data, hardware?
   Was the gain worth it?
4. Is there a row where an older, simpler model is still the best choice?

In [ ]:
# Notes for the current week - free text, keep appending.
#
# Week:
# Best comparable model so far:
# Did this week improve on it?
# Was the extra complexity worth it?

## 6. End-Of-Course Takeaway

Once most weeks are recorded, this table is the honest summary of the semester.
Use it to answer the question the course is really about:

> Given a new dataset and a task, which model would you reach for first, and what
> would you need to see before reaching for something more complicated?

Write your answer here, referring to specific rows and numbers in the table
above.

## 7. Maintenance

`save_results` replaces rows that share the same
(week, dataset, task_type, target, split, model, metric) key, so re-running a
weekly notebook does not create duplicates.

If you do need to reset something, uncomment the relevant line below.

In [ ]:
# clear_benchmark("W05")   # remove one week's rows
# clear_benchmark()        # remove everything and start over